In [ ]:
import ee
import geemap
import geopandas as gpd
import pandas as pd


# Step 0: Initializing GEE

ee.Authenticate()
ee.Initialize(project='heat-wave-483115')


# Step 1: Loading the Sousse ROI (AOI)

delegations = gpd.read_file("/content/sousse.geojson").to_crs(epsg=4326)
governorate_geom = delegations.geometry.union_all()

if governorate_geom.is_empty:
    raise ValueError("Combined governorate geometry is empty. Check your 'sousse.geojson' file.")


governorate_proj = gpd.GeoSeries([governorate_geom], crs="EPSG:4326").to_crs(epsg=32632)
governorate_simplified = governorate_proj.simplify(30).to_crs(epsg=4326).iloc[0]

geojson_data = governorate_simplified.__geo_interface__
if geojson_data['type'] == 'Polygon':
    geojson_data['coordinates'] = [[[c[0], c[1]] for c in ring] for ring in geojson_data['coordinates']]
elif geojson_data['type'] == 'MultiPolygon':
    geojson_data['coordinates'] = [[[[c[0], c[1]] for c in ring] for ring in poly] for poly in geojson_data['coordinates']]

aoi = ee.Geometry(geojson_data)
print("AOI ready.")

AOI ready.


In [ ]:
# Step 2: Building the feature image (Sentinel-2 bands + spectral indices)

s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(aoi)
    .filterDate("2023-01-01", "2023-12-31")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
    .median()
    .clip(aoi)
)

ndvi = s2.normalizedDifference(["B8", "B4"]).rename("NDVI")
ndbi = s2.normalizedDifference(["B11", "B8"]).rename("NDBI")
ndwi = s2.normalizedDifference(["B3", "B8"]).rename("NDWI")
features_img = s2.select(["B2", "B3", "B4", "B8", "B11", "B12"]).addBands([ndvi, ndbi, ndwi])

BANDS = ["B2", "B3", "B4", "B8", "B11", "B12", "NDVI", "NDBI", "NDWI"]


# Step 3: Sampling training data — Source 1: ESA WorldCover (whole governorate, both classes)

worldcover = ee.ImageCollection("ESA/WorldCover/v200").first().clip(aoi)
label_img = worldcover.select("Map").eq(50).rename("built_up")
training_stack = features_img.addBands(label_img)

esa_samples = training_stack.stratifiedSample(
    numPoints=1000,
    classBand="built_up",
    region=aoi,
    scale=10,
    seed=42,
    geometries=True,
)

In [ ]:
# Step 4: Sampling training data — Source 2: own infrastructure evidence
# (lamps + streets = confirmed built_up=1, real ground truth)


lamp_gdf = gpd.read_file("/content/sousse_lamps.geojson").to_crs(epsg=4326)
streets_gdf = gpd.read_file("/content/sousse_streets.geojson").to_crs(epsg=4326)

lamp_buffered = lamp_gdf.to_crs(epsg=32632).buffer(15).to_crs(epsg=4326)
streets_buffered = streets_gdf.to_crs(epsg=32632).buffer(10).to_crs(epsg=4326)

own_evidence_geom = gpd.GeoSeries(
    pd.concat([lamp_buffered, streets_buffered], ignore_index=True)
).union_all()


own_evidence_simplified = gpd.GeoSeries([own_evidence_geom], crs="EPSG:4326") \
    .to_crs(epsg=32632).simplify(5).to_crs(epsg=4326).iloc[0]

own_evidence_ee = ee.Geometry(own_evidence_simplified.__geo_interface__)

own_samples_raw = features_img.addBands(ee.Image.constant(1).rename("built_up")).sample(
    region=own_evidence_ee,
    scale=10,
    numPixels=1000,
    seed=42,
    geometries=True,
)

In [ ]:
# Step 5: Combining training data and training the Random Forest (baseline)

training_samples = esa_samples.merge(own_samples_raw)

classifier = ee.Classifier.smileRandomForest(numberOfTrees=300).train(
    features=training_samples,
    classProperty="built_up",
    inputProperties=BANDS,
)

# Step 5b: Splitting each source separately (so both are represented in test)

esa_samples = esa_samples.randomColumn("random", seed=1)
esa_train = esa_samples.filter(ee.Filter.lt("random", 0.7))
esa_test = esa_samples.filter(ee.Filter.gte("random", 0.7))

own_samples_raw = own_samples_raw.randomColumn("random", seed=2)
own_train = own_samples_raw.filter(ee.Filter.lt("random", 0.7))
own_test = own_samples_raw.filter(ee.Filter.gte("random", 0.7))


# Step 5c: Merging into one combined train set and one combined test set

train_set = esa_train.merge(own_train)
test_set = esa_test.merge(own_test)   # <-- this is the "overall" test set

print("Train size:", train_set.size().getInfo())
print("Test size (combined, both sources):", test_set.size().getInfo())


# Step 5d: Retraining the Random Forest on the combined training set

classifier = ee.Classifier.smileRandomForest(numberOfTrees=300).train(
    features=train_set,
    classProperty="built_up",
    inputProperties=BANDS,
)


# Step 5e: Classifying the combined test set and computing overall accuracy

test_classified = test_set.classify(classifier)

confusion_matrix = test_classified.errorMatrix("built_up", "classification")

print("Confusion Matrix (rows=actual, cols=predicted):")
print(confusion_matrix.getInfo())

print("Overall Accuracy (ESA + own samples combined):", confusion_matrix.accuracy().getInfo())
print("Kappa Coefficient:", confusion_matrix.kappa().getInfo())
print("Producer's Accuracy per class:", confusion_matrix.producersAccuracy().getInfo())
print("Consumer's Accuracy per class:", confusion_matrix.consumersAccuracy().getInfo())

# Step 6: Applying the trained model to every pixel (classifying the full 2023 image)

classified = features_img.select(BANDS).classify(classifier).rename("built_up")


Train size: 2083
Test size (combined, both sources): 917
Confusion Matrix (rows=actual, cols=predicted):
[[270, 32], [34, 581]]
Overall Accuracy (ESA + own samples combined): 0.9280261723009815


/usr/local/lib/python3.12/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Kappa Coefficient: 0.8373440548690357
Producer's Accuracy per class: [[0.8940397350993378], [0.9447154471544715]]
Consumer's Accuracy per class: [[0.8881578947368421, 0.9477977161500816]]


In [ ]:
# Step 7: Exporting the 2023 built-up raster to Google Drive

export_task = ee.batch.Export.image.toDrive(
    image=classified,
    description='sousse_built_up_prediction',
    folder='EarthEngine_exports',
    fileNamePrefix='sousse_built_up_prediction',
    region=aoi,
    scale=10,
    maxPixels=1e10,
    fileFormat='GeoTIFF',
)
export_task.start()
print("Export started. Check status with export_task.status()")

Export started. Check status with export_task.status()


In [ ]:
# Step 8: Converting the 2023 built-up TIF into polygons (shapefile)

import rasterio
from rasterio.features import shapes
import geopandas as gpd
from shapely.geometry import shape

with rasterio.open("/content/sousse_built_up_prediction (1).tif") as src:
    image = src.read(1)  # band 1
    transform = src.transform
    crs = src.crs

    results = (
        {"properties": {"built_up": int(v)}, "geometry": s}
        for s, v in shapes(image, mask=(image == 1), transform=transform)
    )

    geoms = list(results)

gdf = gpd.GeoDataFrame.from_features(geoms, crs=crs)
gdf.to_file("sousse_built_up.shp")
print(f"Saved {len(gdf)} polygons to sousse_built_up.shp")

In [ ]:
# Step 9: Checking 2017 Sentinel-2 image availability before committing

s2_2017_check = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(aoi)
    .filterDate("2017-01-01", "2017-12-31")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
)
print("2017 low-cloud scene count:", s2_2017_check.size().getInfo())

2017 low-cloud scene count: 252


In [ ]:
# Step 10: Building the 2017 feature stack (selecting bands before median()
# to avoid the "incompatible bands" error) and classifying it

NEEDED_BANDS = ["B2", "B3", "B4", "B8", "B11", "B12"]

s2_2017 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(aoi)
    .filterDate("2017-01-01", "2017-12-31")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
    .select(NEEDED_BANDS)
    .median()
    .clip(aoi)
)

ndvi_2017 = s2_2017.normalizedDifference(["B8", "B4"]).rename("NDVI")
ndbi_2017 = s2_2017.normalizedDifference(["B11", "B8"]).rename("NDBI")
ndwi_2017 = s2_2017.normalizedDifference(["B3", "B8"]).rename("NDWI")
features_2017 = s2_2017.addBands([ndvi_2017, ndbi_2017, ndwi_2017])

built_2017 = features_2017.select(BANDS).classify(classifier).rename("built_up")

print("built_2017 ready.")

built_2017 ready.


In [ ]:
# Step 11: Exporting the 2017 built-up raster to Google Drive

export_2017_task = ee.batch.Export.image.toDrive(
    image=built_2017,
    description="sousse_built_2017",
    folder="EarthEngine_exports",
    fileNamePrefix="sousse_built_2017",
    region=aoi,
    scale=10,
    maxPixels=1e10,
    fileFormat="GeoTIFF",
)
export_2017_task.start()
print("Export started. Check status with export_2017_task.status()")

Export started. Check status with export_2017_task.status()


In [ ]:
# Step 12: Converting the 2017 built-up TIF into polygons (shapefile)

import rasterio
from rasterio.features import shapes
import geopandas as gpd
import numpy as np
from shapely.geometry import shape

with rasterio.open("/content/sousse_built_2017.tif") as src:
    image = src.read(1)
    transform = src.transform
    crs = src.crs

    results = (
        {"properties": {"built_up": int(v)}, "geometry": s}
        for s, v in shapes(image, mask=(image == 1), transform=transform)
    )
    geoms = list(results)

built_2017_gdf = gpd.GeoDataFrame.from_features(geoms, crs=crs)
built_2017_gdf.geometry = built_2017_gdf.geometry.simplify(5)  # cleaner edges for web display
built_2017_gdf = built_2017_gdf.to_crs(epsg=4326)  # web maps need WGS84

built_2017_gdf.to_file("/content/sousse_built_2017.shp")
print(f"2017: {len(built_2017_gdf)} polygons")

2017: 216133 polygons


In [ ]:
# Step 13: Loading and aligning the 2017 & 2023 rasters onto one common grid

import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.features import rasterize
import numpy as np

path_2023 = "/content/sousse_built_up_prediction (1).tif"
path_2017 = "/content/sousse_built_2017.tif"

with rasterio.open(path_2023) as src_2023:
    profile_2023 = src_2023.profile
    src_crs = src_2023.crs
    print("2023 raster CRS:", src_crs, "| shape:", src_2023.shape)

# GEE exports default to EPSG:4326 (degrees) unless you set a CRS explicitly.
# For accurate metric distances (meters), reproject both rasters to UTM 32632,
# using the 2023 raster's grid as the reference so both align exactly.
dst_crs = "EPSG:32632"
transform, width, height = calculate_default_transform(
    src_crs, dst_crs, profile_2023["width"], profile_2023["height"],
    *rasterio.open(path_2023).bounds, resolution=10
)

def reproject_to_utm(path, ref_transform, ref_width, ref_height, ref_crs):
    with rasterio.open(path) as src:
        dst_array = np.zeros((ref_height, ref_width), dtype=np.uint8)
        reproject(
            source=rasterio.band(src, 1),
            destination=dst_array,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref_transform,
            dst_crs=ref_crs,
            resampling=Resampling.nearest,
        )
    return dst_array

built_2023 = reproject_to_utm(path_2023, transform, width, height, dst_crs)
built_2017 = reproject_to_utm(path_2017, transform, width, height, dst_crs)

print("Aligned shape:", built_2023.shape, built_2017.shape)
print("2023 built-up pixels:", (built_2023 == 1).sum())
print("2017 built-up pixels:", (built_2017 == 1).sum())


ref_transform, ref_crs, ref_shape = transform, dst_crs, (height, width)
pixel_size_m = 10

2023 raster CRS: EPSG:4326 | shape: (11292, 5904)
Aligned shape: (11322, 4952) (11322, 4952)
2023 built-up pixels: 4842782
2017 built-up pixels: 4444205


In [ ]:
# Step 13a: Extracting the 2017 → 2023 built-up change map (TIF + shapefile)

import numpy as np
import rasterio
from rasterio.features import shapes
import geopandas as gpd

# Build a single categorical raster encoding the comparison between years:
#   0 = stable non-built-up   (empty in both 2017 and 2023)
#   1 = expansion             (empty in 2017, built-up in 2023  -> NEW growth)
#   2 = loss                  (built-up in 2017, empty in 2023  -> usually classification noise)
#   3 = stable built-up       (built-up in both years)
change_map = np.zeros(built_2017.shape, dtype=np.uint8)
change_map[(built_2017 == 0) & (built_2023 == 1)] = 1  # expansion
change_map[(built_2017 == 1) & (built_2023 == 0)] = 2  # loss
change_map[(built_2017 == 1) & (built_2023 == 1)] = 3  # stable built-up

print("Stable non-built-up:", (change_map == 0).sum())
print("Expansion (new):", (change_map == 1).sum())
print("Loss (noise):", (change_map == 2).sum())
print("Stable built-up:", (change_map == 3).sum())

# --- Save as GeoTIFF ---
out_profile = {
    "driver": "GTiff",
    "height": ref_shape[0],
    "width": ref_shape[1],
    "count": 1,
    "dtype": "uint8",
    "crs": ref_crs,
    "transform": ref_transform,
}
with rasterio.open("/content/sousse_built_up_change_2017_2023.tif", "w", **out_profile) as dst:
    dst.write(change_map, 1)
print("Change TIF saved: /content/sousse_built_up_change_2017_2023.tif")

# --- Save as shapefile (vector polygons, only the "changed/built" classes — skips class 0) ---
results = (
    {"properties": {"change_class": int(v)}, "geometry": s}
    for s, v in shapes(change_map, mask=(change_map > 0), transform=ref_transform)
)
geoms = list(results)
change_gdf = gpd.GeoDataFrame.from_features(geoms, crs=ref_crs)

class_labels = {1: "expansion", 2: "loss", 3: "stable_built_up"}
change_gdf["change_label"] = change_gdf["change_class"].map(class_labels)
change_gdf = change_gdf.to_crs(epsg=4326)  # WGS84, standard for sharing/web use

change_gdf.to_file("/content/sousse_built_up_change_2017_2023.shp")
print(f"Change shapefile saved: {len(change_gdf)} polygons")

In [ ]:
# Step 14: Detecting change — building the expansion mask (2017 to 2023)
expansion_mask = (built_2017 == 0) & (built_2023 == 1)
stable_nonbuilt_mask = (built_2017 == 0) & (built_2023 == 0)
candidate_zone = (built_2017 == 0)

print("Expanded pixels:", expansion_mask.sum())
print("Stable non-built pixels:", stable_nonbuilt_mask.sum())

# Step 15: Computing distance to existing 2017 built-up (a driver variable)
from scipy.ndimage import distance_transform_edt

# distance_transform_edt gives distance (in pixels) from each pixel to the
# nearest ZERO-valued pixel in the input. So we invert: 1 = not-built, 0 = built.
dist_to_built = distance_transform_edt(built_2017 != 1) * pixel_size_m

# Step 16: Recreating cleaned street/lamp GeoDataFrames for the local workflow
import geopandas as gpd

streets_gdf = gpd.read_file("/content/sousse_streets.geojson").to_crs(epsg=4326)
lamp_gdf = gpd.read_file("/content/sousse_lamps.geojson").to_crs(epsg=4326)

def clean_for_local(gdf):
    gdf = gdf[gdf.geometry.notnull()]
    gdf = gdf[~gdf.geometry.is_empty]

    poly_types = ["Polygon", "MultiPolygon"]
    is_poly = gdf.geometry.geom_type.isin(poly_types)
    gdf.loc[is_poly, "geometry"] = gdf.loc[is_poly, "geometry"].buffer(0)

    gdf = gdf[gdf.geometry.is_valid]
    gdf = gdf[~gdf.geometry.is_empty]
    return gdf.reset_index(drop=True)

streets_gdf_clean = clean_for_local(streets_gdf)
lamp_gdf_clean = clean_for_local(lamp_gdf)

print("Streets:", len(streets_gdf), "→", len(streets_gdf_clean), "after cleaning")
print("Lamps:", len(lamp_gdf), "→", len(lamp_gdf_clean), "after cleaning")

# Step 17: Rasterizing roads and lamps, then computing their distance layers
streets_utm = streets_gdf_clean.to_crs(ref_crs)
lamps_utm = lamp_gdf_clean.to_crs(ref_crs)

roads_raster = rasterize(
    [(geom, 1) for geom in streets_utm.geometry],
    out_shape=ref_shape, transform=ref_transform, fill=0, dtype=np.uint8,
)
lamps_raster = rasterize(
    [(geom, 1) for geom in lamps_utm.geometry],
    out_shape=ref_shape, transform=ref_transform, fill=0, dtype=np.uint8,
)

dist_to_roads = distance_transform_edt(roads_raster != 1) * pixel_size_m
dist_to_lamps = distance_transform_edt(lamps_raster != 1) * pixel_size_m

# Step 18: Computing distance to the city center (pixel-weighted 2017 centroid)
rows, cols = np.where(built_2017 == 1)
center_row, center_col = rows.mean(), cols.mean()
center_x, center_y = rasterio.transform.xy(ref_transform, center_row, center_col)

grid_rows, grid_cols = np.indices(ref_shape)
grid_x, grid_y = rasterio.transform.xy(ref_transform, grid_rows, grid_cols)
grid_x, grid_y = np.array(grid_x), np.array(grid_y)

dist_to_center = np.sqrt((grid_x - center_x) ** 2 + (grid_y - center_y) ** 2)

print("City center (UTM):", center_x, center_y)

# Step 19: Leaving slope as a placeholder here (filled in once the DEM is ready)
# slope = ...  (see Step 10b)

print("Driver layers ready (except slope).")

Expanded pixels: 1907449
Stable non-built pixels: 49714890
Streets: 3490 → 3456 after cleaning
Lamps: 15764 → 15764 after cleaning
City center (UTM): 629761.4647817257 3978394.4702731576
Driver layers ready (except slope).


In [ ]:
# Step 20: Exporting the SRTM DEM (needed for the slope driver)

# Run once in GEE, download the tif to /content like the others:
dem_export = ee.batch.Export.image.toDrive(
    image=ee.Image("USGS/SRTMGL1_003").clip(aoi),
    description="sousse_dem",
    folder="EarthEngine_exports",
    fileNamePrefix="sousse_dem",
    region=aoi,
    scale=30,
    maxPixels=1e10,
    fileFormat="GeoTIFF",
)
dem_export.start()

/usr/local/lib/python3.12/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


In [ ]:
# Step 21: Computing slope from the DEM

dem = reproject_to_utm("/content/sousse_dem.tif", ref_transform, width, height, dst_crs).astype(float)

gy, gx = np.gradient(dem, pixel_size_m)
slope_rad = np.arctan(np.sqrt(gx**2 + gy**2))
slope = np.degrees(slope_rad)

print("Slope ready.")

Slope ready.


In [ ]:
# Step 22: Checking driver layer shapes before stacking them
print("dist_to_built:", dist_to_built.shape)
print("dist_to_roads:", dist_to_roads.shape)
print("dist_to_lamps:", dist_to_lamps.shape)
print("dist_to_center:", dist_to_center.shape)
print("slope:", slope.shape)
print("Reference shape (ref_shape):", ref_shape)

dist_to_built: (11322, 4952)
dist_to_roads: (11322, 4952)
dist_to_lamps: (11322, 4952)
dist_to_center: (56066544,)
slope: (11322, 4952)
Reference shape (ref_shape): (11322, 4952)


In [ ]:
# Step 23: Fixing and finalizing the dist_to_center shape
grid_rows, grid_cols = np.indices(ref_shape)
grid_x, grid_y = rasterio.transform.xy(ref_transform, grid_rows, grid_cols)
grid_x = np.array(grid_x).reshape(ref_shape)   # <-- added reshape
grid_y = np.array(grid_y).reshape(ref_shape)   # <-- added reshape

dist_to_center = np.sqrt((grid_x - center_x) ** 2 + (grid_y - center_y) ** 2)

print("dist_to_center shape (should match ref_shape):", dist_to_center.shape, "vs", ref_shape)

dist_to_center shape (should match ref_shape): (11322, 4952) vs (11322, 4952)


In [ ]:
# Step 24: Building the driver stack and sampling training points

DRIVER_STACK = np.stack([dist_to_built, dist_to_roads, dist_to_lamps, dist_to_center, slope], axis=-1)
DRIVERS = ["dist_built", "dist_roads", "dist_lamps", "dist_center", "slope"]

candidate_idx = np.where(candidate_zone)
labels_at_candidates = expansion_mask[candidate_idx]

expanded_idx = np.where(labels_at_candidates == 1)[0]
stable_idx = np.where(labels_at_candidates == 0)[0]

rng = np.random.default_rng(42)
n_per_class = min(750, len(expanded_idx), len(stable_idx))  # stratified, capped like before

sel_expanded = rng.choice(expanded_idx, size=n_per_class, replace=False)
sel_stable = rng.choice(stable_idx, size=n_per_class, replace=False)
sel_all = np.concatenate([sel_expanded, sel_stable])

rows_sel = candidate_idx[0][sel_all]
cols_sel = candidate_idx[1][sel_all]

X = DRIVER_STACK[rows_sel, cols_sel, :]
y = labels_at_candidates[sel_all]

print("Sample size:", X.shape, "| Class balance:", y.sum(), "/", len(y))

Sample size: (1500, 5) | Class balance: 750 / 1500


In [ ]:
# Step 25: Splitting the data and training the expansion Random Forest

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=7, stratify=y
)

expansion_clf = RandomForestClassifier(n_estimators=300, random_state=42)
expansion_clf.fit(X_train, y_train)

print("Model trained.")

Model trained.


In [ ]:
# Step 26: Evaluating the expansion model (accuracy + feature importance)

from sklearn.metrics import confusion_matrix, accuracy_score, cohen_kappa_score, classification_report

y_pred = expansion_clf.predict(X_test)

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Overall Accuracy:", accuracy_score(y_test, y_pred))
print("Kappa:", cohen_kappa_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["stable", "expanded"]))

importances = dict(zip(DRIVERS, expansion_clf.feature_importances_))
print("Driver importance:", importances)

Confusion Matrix:
 [[184  41]
 [ 29 196]]
Overall Accuracy: 0.8444444444444444
Kappa: 0.6888888888888889
              precision    recall  f1-score   support

      stable       0.86      0.82      0.84       225
    expanded       0.83      0.87      0.85       225

    accuracy                           0.84       450
   macro avg       0.85      0.84      0.84       450
weighted avg       0.85      0.84      0.84       450

Driver importance: {'dist_built': np.float64(0.4488170708850549), 'dist_roads': np.float64(0.1350144467311733), 'dist_lamps': np.float64(0.12891007930053947), 'dist_center': np.float64(0.16083931786372194), 'slope': np.float64(0.12641908521951054)}


In [ ]:
# Step 27: Counting candidate pixels to predict

flat_stack = DRIVER_STACK.reshape(-1, len(DRIVERS))
valid_mask = candidate_zone.flatten()
valid_indices = np.where(valid_mask)[0]

print("Total pixels:", flat_stack.shape[0])
print("Valid (candidate) pixels to predict:", len(valid_indices))

Total pixels: 56066544
Valid (candidate) pixels to predict: 51622339


In [ ]:
# Step 28: Checkpointing the trained model and driver stack

import joblib

joblib.dump(expansion_clf, "/content/expansion_clf.pkl")
np.save("/content/DRIVER_STACK.npy", DRIVER_STACK.astype(np.float32))
np.save("/content/candidate_zone.npy", candidate_zone)
np.save("/content/ref_transform.npy", np.array(ref_transform))
print("Checkpoint saved.")

Checkpoint saved.


In [ ]:
# Step 29: Reloading the checkpointed model and driver stack

import numpy as np, rasterio, gc, joblib

expansion_clf = joblib.load("/content/expansion_clf.pkl")
DRIVER_STACK = np.load("/content/DRIVER_STACK.npy")          # already float32
candidate_zone = np.load("/content/candidate_zone.npy")
ref_shape = candidate_zone.shape

print("Reloaded. DRIVER_STACK dtype:", DRIVER_STACK.dtype, "shape:", DRIVER_STACK.shape)

Reloaded. DRIVER_STACK dtype: float32 shape: (11322, 4952, 5)


In [ ]:
# Step 30: Predicting expansion suitability in batches

flat_stack = DRIVER_STACK.reshape(-1, len(["dist_built","dist_roads","dist_lamps","dist_center","slope"]))
valid_mask = candidate_zone.flatten()
valid_indices = np.where(valid_mask)[0]

suitability_flat = np.zeros(flat_stack.shape[0], dtype=np.float32)

batch_size = 100_000   # smaller than before — 500k was likely too much combined with everything else in RAM
n_batches = int(np.ceil(len(valid_indices) / batch_size))

for i in range(n_batches):
    start = i * batch_size
    end = min((i + 1) * batch_size, len(valid_indices))
    batch_idx = valid_indices[start:end]

    batch_pred = expansion_clf.predict_proba(flat_stack[batch_idx])[:, 1]
    suitability_flat[batch_idx] = batch_pred

    del batch_pred
    if i % 20 == 0:
        gc.collect()
        print(f"Batch {i+1}/{n_batches} done")

suitability = suitability_flat.reshape(ref_shape)
print("Prediction complete.")

np.save("/content/suitability.npy", suitability)  # checkpoint the result immediately
print("Suitability checkpointed.")

Batch 1/517 done
Batch 21/517 done
Batch 41/517 done
Batch 61/517 done
Batch 81/517 done
Batch 101/517 done
Batch 121/517 done
Batch 141/517 done
Batch 161/517 done
Batch 181/517 done
Batch 201/517 done
Batch 221/517 done
Batch 241/517 done
Batch 261/517 done
Batch 281/517 done
Batch 301/517 done
Batch 321/517 done
Batch 341/517 done
Batch 361/517 done
Batch 381/517 done
Batch 401/517 done
Batch 421/517 done
Batch 441/517 done
Batch 461/517 done
Batch 481/517 done
Batch 501/517 done
Prediction complete.
Suitability checkpointed.


In [ ]:
# Step 31: Recomputing the grid transform directly from the source TIF

import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform

# Recompute the exact same alignment used in Step 9, directly from the source tif
# Adjust this filename if your 2023 tif has a different name in /content
path_2023 = "/content/sousse_built_up_prediction (1).tif"

with rasterio.open(path_2023) as src_2023:
    src_crs = src_2023.crs
    dst_crs = "EPSG:32632"
    ref_transform, width, height = calculate_default_transform(
        src_crs, dst_crs, src_2023.width, src_2023.height, *src_2023.bounds, resolution=10
    )

ref_shape = (height, width)
ref_crs = dst_crs

print("ref_transform type:", type(ref_transform))
print("ref_transform:", ref_transform)
print("ref_shape:", ref_shape)

# Load the checkpointed suitability prediction
suitability = np.load("/content/suitability.npy")
print("suitability shape:", suitability.shape)

ref_transform type: <class 'affine.Affine'>
ref_transform: | 10.00, 0.00, 603824.27|
| 0.00,-10.00, 4028732.96|
| 0.00, 0.00, 1.00|
ref_shape: (11322, 4952)
suitability shape: (11322, 4952)


In [ ]:
# Step 32: Saving the suitability raster (final, self-contained and validated)
ref_crs = "EPSG:32632"   # explicit, plain string — re-set here to be safe

# Sanity checks before writing
assert isinstance(ref_crs, str), f"ref_crs is not a string! type={type(ref_crs)}, value={ref_crs}"
assert isinstance(ref_transform, Affine), f"ref_transform is not an Affine! type={type(ref_transform)}"
assert suitability.shape == ref_shape, f"Shape mismatch: {suitability.shape} vs {ref_shape}"

print("ref_crs:", ref_crs, type(ref_crs))
print("ref_transform:", ref_transform, type(ref_transform))
print("suitability dtype:", suitability.dtype)

out_profile = {
    "driver": "GTiff",
    "height": int(ref_shape[0]),
    "width": int(ref_shape[1]),
    "count": 1,
    "dtype": "float32",
    "crs": ref_crs,
    "transform": ref_transform,
}

print("out_profile:", out_profile)

with rasterio.open("/content/sousse_expansion_suitability.tif", "w", **out_profile) as dst:
    dst.write(suitability.astype(np.float32), 1)

print("Suitability raster saved locally.")

import os
print("File exists:", os.path.exists("/content/sousse_expansion_suitability.tif"))

ref_crs: EPSG:32632 <class 'str'>
ref_transform: | 10.00, 0.00, 603824.27|
| 0.00,-10.00, 4028732.96|
| 0.00, 0.00, 1.00| <class 'affine.Affine'>
suitability dtype: float32
out_profile: {'driver': 'GTiff', 'height': 11322, 'width': 4952, 'count': 1, 'dtype': 'float32', 'crs': 'EPSG:32632', 'transform': Affine(10.0, 0.0, 603824.2703329998,
       0.0, -10.0, 4028732.960566697)}
Suitability raster saved locally.
File exists: True
